# RAGBench Capstone — Legal Domain (cuad)
**Batch 26 | EXP-001 Baseline**

Same frame as Customer Support / Finance notebooks (Universal Pipeline skeleton).

**Locked conventions (do not change):**
- SEED=42, k=5 retrieval, timeout=30s, N=25 iteration → N=200 finals
- Judge = `llama-3.3-70b-versatile` (never downgrade to 8b)
- Judge prompt = **verbatim Appendix 7.4 (Friel et al. 2024) — PASTE MANUALLY in Cell 8, never machine-generated**
- `.format(documents=, question=, answer=)` exactly
- Documents to judge as raw nested `[[key, sentence], ...]`, renumbered 0,1,2… in retrieval order
- Judge keys clipped against real retrieved keys
- `overall_supported` fallback for adherence
- Corrected TRACe: utilization = |utilized|/total, completeness = |relevant∩utilized|/|relevant|
- Triple-safety checkpointing every 5 examples (Colab local + Drive + GitHub)
- Smoke test 3 examples with full error visibility BEFORE full run

## Cell 1 — Config (single source of truth)

In [1]:
# ============ EXPERIMENT CONFIG ============
CONFIG = {
    "experiment_id":   "LEGAL-EXP-001",
    "domain":          "legal",
    "dataset_config":  "cuad",                      # RAGBench subset
    "split":           "test",

    # --- pipeline components (baseline) ---
    "chunking":        "sentence",                  # sentence | sliding | fixed | whole_doc | metadata
    "chunk_size":      3,                            # sentences per chunk (sentence mode)
    "chunk_overlap":   1,
    "embedder":        "sentence-transformers/all-MiniLM-L6-v2",
    "retrieval":       "dense_faiss",                # dense_faiss | hybrid_rrf
    "reranker":        None,                         # None | cross-encoder | ...
    "gen_model":       "llama-3.3-70b-versatile",
    "judge_model":     "llama-3.3-70b-versatile",   # LOCKED — 70b essential

    # --- run params (locked) ---
    "SEED":            42,
    "N":               25,                           # 25 for iteration, 200 for finals
    "k":               5,
    "timeout":         30,
    "checkpoint_every": 5,

    # --- paths ---
    "drive_dir":  "/content/drive/MyDrive/RAGBench_Capstone/legal",
    "local_dir":  "/content/results/legal",
    "repo":       "veenulearns-lab/RAGBench-Capstone-Batch26",
    "repo_path":  "results/legal",
}

import os, json, random
import numpy as np
random.seed(CONFIG["SEED"]); np.random.seed(CONFIG["SEED"])
os.makedirs(CONFIG["local_dir"], exist_ok=True)
print(f"Config loaded: {CONFIG['experiment_id']} | N={CONFIG['N']} | {CONFIG['chunking']} + {CONFIG['embedder'].split('/')[-1]} + {CONFIG['retrieval']} + {CONFIG['gen_model']}")

Config loaded: LEGAL-EXP-001 | N=25 | sentence + all-MiniLM-L6-v2 + dense_faiss + llama-3.3-70b-versatile


## Cell 2 — Installs

In [2]:
!pip install -q datasets sentence-transformers faiss-cpu groq nltk rank_bm25
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print("Installs done.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.7 MB/s eta 0:00:00
Installs done.


## Cell 3 — Drive mount + GitHub setup (triple safety)

In [3]:
from google.colab import drive, userdata
drive.mount('/content/drive')
os.makedirs(CONFIG["drive_dir"], exist_ok=True)

GH_PAT = userdata.get('GIT_PAT_V')
REPO_URL = f"https://{GH_PAT}@github.com/{CONFIG['repo']}.git"

if not os.path.exists('/content/repo'):
    !git clone {REPO_URL} /content/repo -q
!cd /content/repo && git config user.email "veenulearns-lab@users.noreply.github.com" && git config user.name "veenulearns-lab"
os.makedirs(f"/content/repo/{CONFIG['repo_path']}", exist_ok=True)
print("Drive + GitHub ready.")

Mounted at /content/drive
Drive + GitHub ready.


## Cell 4 — Groq 5-key rotation

In [4]:
import time
from groq import Groq

KEYS = [userdata.get(f'GROQ_API_KEY_{i}') for i in range(1, 6)]
KEYS = [k for k in KEYS if k]
assert len(KEYS) == 5, f"Expected 5 keys, got {len(KEYS)} — check Colab secrets"
_key_idx = 0

def groq_call(messages, model, max_retries=10):
    """Rotate keys on 429. NEVER silently swallow non-429 exceptions."""
    global _key_idx
    for attempt in range(max_retries):
        try:
            client = Groq(api_key=KEYS[_key_idx], timeout=CONFIG["timeout"])
            resp = client.chat.completions.create(
                model=model, messages=messages, temperature=0.0
            )
            return resp.choices[0].message.content
        except Exception as e:
            err = str(e)
            if '429' in err or 'rate limit' in err.lower():
                _key_idx = (_key_idx + 1) % len(KEYS)
                print(f"  [429] rotating to key {_key_idx + 1}, attempt {attempt + 1}")
                time.sleep(3)
            else:
                # Full visibility — re-raise everything that isn't a rate limit
                print(f"  [NON-429 ERROR] {type(e).__name__}: {err[:300]}")
                raise
    raise RuntimeError(f"All {max_retries} retries exhausted across keys")

print(f"{len(KEYS)} Groq keys loaded, rotation ready.")

5 Groq keys loaded, rotation ready.


In [5]:
#HFtoken
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

## Cell 5 — Load cuad + deterministic sample

In [6]:
from datasets import load_dataset

ds = load_dataset("rungalileo/ragbench", CONFIG["dataset_config"], split=CONFIG["split"])
print(f"cuad {CONFIG['split']} split: {len(ds)} examples")

# Deterministic sample — same indices every run at SEED=42
rng = random.Random(CONFIG["SEED"])
sample_indices = sorted(rng.sample(range(len(ds)), min(CONFIG["N"], len(ds))))
sample = ds.select(sample_indices)
print(f"Sampled N={len(sample)} | first 5 indices: {sample_indices[:5]}")

# Quick shape check — cuad docs are long contracts
ex0 = sample[0]
print(f"\nFields: {list(ex0.keys())[:10]}")
print(f"Num documents in ex0: {len(ex0['documents'])}")
print(f"Doc 0 length (chars): {len(ex0['documents'][0])}")

README.md:   0%|          | 0.00/24.7k [00:00<?, ?B/s]

cuad/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 56.4MB            

cuad/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

cuad/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 15.7MB            

cuad/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

cuad/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/510 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/510 [00:00<?, ? examples/s]

cuad test split: 510 examples
Sampled N=25 | first 5 indices: [12, 13, 15, 16, 44]

Fields: ['id', 'question', 'documents', 'response', 'generation_model_name', 'annotating_model_name', 'dataset_name', 'documents_sentences', 'response_sentences', 'sentence_support_information']
Num documents in ex0: 1
Doc 0 length (chars): 70898


## Cell 5b — SAC gate: do cuad docs retain contract structure?
Zero API calls. If markers are stripped, SAC chunking (EXP-004) is dead on arrival — skip it.

In [9]:
# import re

# # Contract structure markers to look for
# PATTERNS = {
#     "section_num":   re.compile(r"\bSection\s+\d+(\.\d+)*\b", re.IGNORECASE),
#     "article":       re.compile(r"\bARTICLE\s+([IVXLC]+|\d+)\b", re.IGNORECASE),
#     "numbered_hdr":  re.compile(r"^\s*\d+(\.\d+)+\s+[A-Z]", re.MULTILINE),
#     "allcaps_hdr":   re.compile(r"^[A-Z][A-Z\s]{8,60}$", re.MULTILINE),   # e.g. GOVERNING LAW
#     "defined_terms": re.compile(r"\(the\s+[\u201c\"][A-Z]"),                # (the \"Agreement\"
#     "exhibit":       re.compile(r"\bEXHIBIT\s+[A-Z0-9]\b", re.IGNORECASE),
# }

# N_CHECK = 5
# print(f"Structure check on {N_CHECK} sampled examples:\n")
# totals = {k: 0 for k in PATTERNS}
# docs_with_any = 0
# n_docs = 0

# for i in range(N_CHECK):
#     ex = sample[i]
#     print(f"--- ex {i} (idx={sample_indices[i]}) | {len(ex['documents'])} docs ---")
#     for d_i, doc in enumerate(ex['documents']):
#         n_docs += 1
#         hits = {k: len(p.findall(doc)) for k, p in PATTERNS.items()}
#         for k, v in hits.items():
#             totals[k] += v
#         any_hit = sum(hits.values()) > 0
#         docs_with_any += any_hit
#         hit_str = ", ".join(f"{k}={v}" for k, v in hits.items() if v) or "NONE"
#         print(f"  doc{d_i} ({len(doc)} chars): {hit_str}")
#     # show first 300 chars of doc0 so you can eyeball formatting/linebreaks
#     print(f"  doc0 head: {repr(ex['documents'][0][:300])}\n")

# print("=" * 50)
# print(f"Docs with ≥1 structure marker: {docs_with_any}/{n_docs}")
# for k, v in totals.items():
#     print(f"  {k:13s}: {v} total hits")

# verdict = docs_with_any / n_docs if n_docs else 0
# if verdict >= 0.6:
#     print("\n✅ GATE PASS — structure retained. SAC chunking (EXP-004) is viable if TRACe points at chunking.")
# elif verdict >= 0.3:
#     print("\n⚠️ PARTIAL — some structure. Eyeball the doc heads above; check if linebreaks survive (allcaps/numbered headers need real newlines).")
# else:
#     print("\n⛔ GATE FAIL — docs look pre-stripped. Drop SAC from the queue; stick to embedder → k sweep.")

Structure check on 5 sampled examples:

--- ex 0 (idx=12) | 1 docs ---
  doc0 (70898 chars): section_num=33, allcaps_hdr=2, defined_terms=5
  doc0 head: 'Exhibit 10.1\n\nCERTAIN CONFIDENTIAL INFORMATION CONTAINED IN THIS DOCUMENT, MARKED BY BRACKETED ASTERISKS [***], HAS BEEN OMITTED AND FILED SEPARATELY WITH THE SECURITIES AND EXCHANGE COMMISSION PURSUANT TO RULE 24B-2 OF THE SECURITIES EXCHANGE ACT OF 1934, AS AMENDED. SUPPLY AGREEMENT\n\nTHIS SUPPLY A'

--- ex 1 (idx=13) | 1 docs ---
  doc0 (23282 chars): section_num=1
  doc0 head: 'Exhibit 10.2   INTELLECTUAL PROPERTY AGREEMENT   This Intellectual Property Agreement (this "Agreement") is entered into on May 12, 2020 ("Effective Date"), concerning the pursuits set forth herein for the collective development, implementation and commercialization of a potential treatment for the '

--- ex 2 (idx=15) | 1 docs ---
  doc0 (8052 chars): defined_terms=1
  doc0 head: 'SPONSORSHIP AGREEMENT This agreement (the "Agreement") is made effective 

## Cell 6 — Chunking + sentence keying
Sentence-keyed structure `[[key, sentence], ...]` per doc — keys renumbered 0,1,2… in retrieval order before the judge sees them.

In [7]:
from nltk.tokenize import sent_tokenize

def chunk_documents(documents, cfg):
    """Returns list of chunks; each chunk = list of sentence strings."""
    chunks = []
    for doc in documents:
        sents = sent_tokenize(doc)
        if cfg["chunking"] == "sentence":
            step = max(1, cfg["chunk_size"] - cfg["chunk_overlap"])
            for i in range(0, len(sents), step):
                grp = sents[i:i + cfg["chunk_size"]]
                if grp:
                    chunks.append(grp)
        elif cfg["chunking"] == "whole_doc":
            chunks.append(sents)
        elif cfg["chunking"] == "sliding":
            # PORT-IN: copy exact sliding chunker verbatim from locked Customer Support notebook if switching
            raise NotImplementedError("Port sliding chunker from CS notebook")
        else:
            raise ValueError(f"Unknown chunking: {cfg['chunking']}")
    return chunks

def keyed_structure(retrieved_chunks):
    """Raw nested [[key, sentence], ...] renumbered 0,1,2... in retrieval order.
    Keys: '0a','0b',... per RAGBench convention (doc index + sentence letter)."""
    def letters(n):
        s, n = "", n + 1
        while n:
            n, r = divmod(n - 1, 26)
            s = chr(97 + r) + s
        return s
    docs = []
    all_keys = []
    for d_i, chunk in enumerate(retrieved_chunks):
        pairs = []
        for s_i, sent in enumerate(chunk):
            key = f"{d_i}{letters(s_i)}"
            pairs.append([key, sent])
            all_keys.append(key)
        docs.append(pairs)
    return docs, set(all_keys)

# sanity
_test = chunk_documents([sample[0]['documents'][0]], CONFIG)
print(f"ex0 doc0 → {len(_test)} chunks | first chunk sentences: {len(_test[0])}")

ex0 doc0 → 204 chunks | first chunk sentences: 3


## Cell 7 — Embedding + FAISS retrieval (k=5)

In [8]:
import faiss
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(CONFIG["embedder"])
EMB_DIM = embedder.get_sentence_embedding_dimension()
print(f"Embedder: {CONFIG['embedder']} | dim={EMB_DIM}")  # guard against stale-dim bug

def retrieve(question, documents, cfg):
    """Per-example index (fresh each time — no stale global embedder state)."""
    chunks = chunk_documents(documents, cfg)
    chunk_texts = [" ".join(c) for c in chunks]
    emb = embedder.encode(chunk_texts, normalize_embeddings=True, show_progress_bar=False)
    assert emb.shape[1] == EMB_DIM, f"Dim mismatch! {emb.shape[1]} vs {EMB_DIM}"
    index = faiss.IndexFlatIP(EMB_DIM)
    index.add(emb.astype('float32'))
    q_emb = embedder.encode([question], normalize_embeddings=True).astype('float32')
    k = min(cfg["k"], len(chunks))
    _, idxs = index.search(q_emb, k)
    return [chunks[i] for i in idxs[0]]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedder: sentence-transformers/all-MiniLM-L6-v2 | dim=384


/tmp/ipykernel_486/2524855021.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  EMB_DIM = embedder.get_sentence_embedding_dimension()


## Cell 8 — ⚠️ JUDGE PROMPT — VEE PASTES MANUALLY ⚠️
**Paste verbatim Appendix 7.4 from Friel et al. 2024 below. Do NOT let any tool generate or edit this string.**

`.format()` placeholders must be exactly: `documents=`, `question=`, `answer=`

In [9]:
JUDGE_PROMPT_TEMPLATE = """I asked someone to answer a question based on one or more
documents. Your task is to review their response and assess whether or not each
sentence in that response is supported by text in the documents. And if so, which
sentences in the documents provide that support. You will also tell me which
of the documents contain useful information for answering the question, and
which of the documents the answer was sourced from.

Here are the documents, each of which is split into sentences. Alongside each
sentence is associated key, such as '0a.' or '0b.' that you can use to refer
to it:

```
{documents}
```

The question was:
```
{question}
```

Here is their response, split into sentences. Alongside each sentence is
associated key, such as 'a.' or 'b.' that you can use to refer to it. Note
that these keys are unique to the response, and are not related to the keys
in the documents:

```
{answer}
```

You must respond with a JSON object matching this schema:

{{
  "relevance_explanation": string,
  "all_relevant_sentence_keys": [string],
  "overall_supported_explanation": string,
  "overall_supported": boolean,
  "sentence_support_information": [
    {{
      "response_sentence_key": string,
      "explanation": string,
      "supporting_sentence_keys": [string],
      "fully_supported": boolean
    }}
  ],
  "all_utilized_sentence_keys": [string]
}}

The relevance_explanation field is a string explaining which documents
contain useful information for answering the question. Walk through the
information in the documents step by step and how it is useful for
answering the question.

The all_relevant_sentence_keys field is a list of all document sentence
keys (e.g. '0a') that are relevant to the question. Include every sentence
that is useful and relevant to the question, even if it was not used in the
response, or if only parts of the sentence are useful. Base this judgement
only on the documents and the question -- ignore the response entirely when
deciding relevance. Leave out sentences that could be removed from the
document without affecting someone's ability to answer the question.

The overall_supported_explanation field is a string explaining why the
response *as a whole* is or is not supported by the documents. Walk through
each claim in the response individually and assess its support (or lack of
support) in the documents one at a time, before drawing any conclusion about
the response as a whole.

The overall_supported field is a boolean reflecting the conclusion you
reached at the end of overall_supported_explanation: whether the response as
a whole is supported by the documents.

The sentence_support_information field is a list of objects, one for each sentence
in the response. Each object MUST have the following fields:
- response_sentence_key: a string identifying the sentence in the response. This
key is the same as the one used in the response above.- explanation: a string
explaining why the sentence is or is not supported by the documents.
- supporting_sentence_keys: keys (e.g. ’0a’) of sentences from the documents that
support the response sentence. If the sentence is not supported, this list MUST
be empty. If the sentence is supported, this list MUST contain one or more keys.
In special cases where the sentence is supported, but not by any specific sentence,
you can use the string "supported_without_sentence" to indicate that the sentence
is generally supported by the documents. Consider cases where the sentence is
expressing inability to answer the question due to lack of relevant information
in the provided contex as "supported_without_sentence". In cases
where the sentence is making a general statement (e.g. outlining the steps to produce
an answer, or summarizing previously stated sentences, or a transition sentence), use
the sting "general".In cases where the sentence is correctly stating a well-known fact,
like a mathematical formula, use the string "well_known_fact". In cases where the
sentence is performing numerical reasoning (e.g. addition, multiplication), use
the string "numerical_reasoning".
- fully_supported: a boolean indicating whether the sentence is fully supported by
the documents.
  - This value should reflect the conclusion  you drew at the end of your step-by-step
    breakdown in explanation.
  - If supporting_sentence_keys is an empty list, then fully_supported must be false.
  - Otherwise, use fully_supported to clarify whether everything in the response
  sentence is fully supported by the document text indicated in supporting_sentence_keys
  (fully_supported = true), or whether the sentence is only partially or incompletely
  supported by that document text (fully_supported = false).

The all_utilized_sentence_keys field is a list of all sentences keys (e.g. ’0a’) that
were used to construct the answer. Include every sentence that either directly supported
the answer, or was implicitly used to construct the answer, even if it was not used
in its entirety. Omit sentences that were not used, and could have been removed from
the documents without affecting the answer.

You must respond with a valid JSON string. Use escapes for quotes, e.g. ‘\\"‘, and
newlines, e.g. ‘\\n‘. Do not write anything before or after the JSON string. Do not
wrap the JSON string in backticks like ‘‘‘ or ‘‘‘json.

As a reminder: your task is to review the response and assess which documents contain
useful information pertaining to the question, and how each sentence in the response
is supported by the text in the documents."""

assert "PASTE APPENDIX" not in JUDGE_PROMPT_TEMPLATE, "⛔ Judge prompt not pasted yet — paste Appendix 7.4 verbatim before running."
for ph in ["{documents}", "{question}", "{answer}"]:
    assert ph in JUDGE_PROMPT_TEMPLATE, f"⛔ Missing placeholder {ph} in judge prompt"
print("Judge prompt loaded and placeholders verified.")

Judge prompt loaded and placeholders verified.


## Cell 9 — Generation + Judge + corrected TRACe

In [10]:
GEN_PROMPT = (
    "Answer the question using only the provided context.\n\n"
    "Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"
)

def generate_answer(question, retrieved_chunks, model=None):
    context = "\n\n".join(" ".join(c) for c in retrieved_chunks)
    return groq_call(
        [{"role": "user", "content": GEN_PROMPT.format(context=context, question=question)}],
        model=model or CONFIG["gen_model"]
    )

def _salvage_judge_fields(raw):
    """Last-resort field extraction when judge JSON is structurally broken (8b failure mode)."""
    import re as _r
    out = {"_salvaged": True}
    for field in ("all_relevant_sentence_keys", "all_utilized_sentence_keys"):
        m = _r.search(field + r'"\s*:\s*\[(.*?)\]', raw, _r.DOTALL)
        out[field] = _r.findall(r'"([^"]+)"', m.group(1)) if m else []
    m = _r.search(r'"overall_supported"\s*:\s*(true|false)', raw, _r.IGNORECASE)
    out["overall_supported"] = (m.group(1).lower() == "true") if m else False
    # sentence_support_information unrecoverable from broken JSON -> omit; adherence falls back to overall_supported
    return out

def parse_judge_json(raw):
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        raw = raw[4:] if raw.startswith("json") else raw
    start, end = raw.find("{"), raw.rfind("}")
    if start == -1 or end == -1:
        raise ValueError(f"No JSON object in judge output: {raw[:200]}")
    s2 = raw[start:end + 1]
    try:
        return json.loads(s2)
    except json.JSONDecodeError:
        pass
    try:   # escape repair
        import re as _re_json
        repaired = _re_json.sub(r'\\(?!["\\/bfnrtu])', r'\\\\', s2)
        return json.loads(repaired)
    except json.JSONDecodeError:
        print("    [judge JSON malformed — salvaging fields]")
        return _salvage_judge_fields(raw)

def judge_and_score(question, answer, retrieved_chunks, judge_model=None):
    docs_struct, real_keys = keyed_structure(retrieved_chunks)
    total_sents = len(real_keys)

    prompt = JUDGE_PROMPT_TEMPLATE.format(
        documents=docs_struct,   # raw nested [[key, sentence], ...]
        question=question,
        answer=answer
    )
    raw = groq_call([{"role": "user", "content": prompt}], model=judge_model or CONFIG["judge_model"])
    j = parse_judge_json(raw)

    # --- clip judge keys against real retrieved keys (anti-hallucination) ---
    relevant = set(j.get("all_relevant_sentence_keys", [])) & real_keys
    utilized = set(j.get("all_utilized_sentence_keys", [])) & real_keys

    # --- adherence with overall_supported fallback ---
    sent_supp = j.get("sentence_support_information", [])
    if sent_supp:
        flags = [s.get("fully_supported", False) for s in sent_supp]
        adherence = 1.0 if all(flags) else 0.0
    else:
        adherence = 1.0 if j.get("overall_supported", False) else 0.0

    # --- corrected TRACe formulas ---
    relevance    = len(relevant) / total_sents if total_sents else 0.0
    utilization  = len(utilized) / total_sents if total_sents else 0.0
    completeness = len(relevant & utilized) / len(relevant) if relevant else 0.0

    return {
        "relevance": relevance, "utilization": utilization,
        "completeness": completeness, "adherence": adherence,
        "judge_raw": j,
        "n_relevant": len(relevant), "n_utilized": len(utilized),
        "n_total_sents": total_sents,
    }


## Cell 10 — Checkpointing (triple safety) + progress-file staleness guard

In [11]:
RESULTS_FILE = f"{CONFIG['experiment_id']}_results.json"

def load_progress():
    path = os.path.join(CONFIG["drive_dir"], RESULTS_FILE)
    if os.path.exists(path):
        with open(path) as f:
            results = json.load(f)
        # staleness guard: only count results matching THIS experiment + sample
        results = [r for r in results if r.get("experiment_id") == CONFIG["experiment_id"]
                   and r.get("dataset_index") in set(sample_indices)]
        done = {r["dataset_index"] for r in results}
        print(f"Resuming: {len(done)} examples already completed for {CONFIG['experiment_id']}")
        return results, done
    return [], set()

def checkpoint(results, push_github=False):
    local = os.path.join(CONFIG["local_dir"], RESULTS_FILE)
    drv   = os.path.join(CONFIG["drive_dir"], RESULTS_FILE)
    for p in (local, drv):
        with open(p, "w") as f:
            json.dump(results, f, indent=1)
    if push_github:
        repo_file = f"/content/repo/{CONFIG['repo_path']}/{RESULTS_FILE}"
        with open(repo_file, "w") as f:
            json.dump(results, f, indent=1)
        !cd /content/repo && git add -A && git commit -m "{CONFIG['experiment_id']}: {len(results)} examples" -q && git push -q
    print(f"  ✓ checkpointed {len(results)} results" + (" + pushed" if push_github else ""))

## Cell 11 — 🔬 SMOKE TEST: 3 examples, full error visibility
**Run this and check output BEFORE the full loop. Do not proceed if anything looks off.**

In [15]:
for i in range(3):
    ex = sample[i]
    print(f"\n{'='*60}\nSMOKE {i+1}/3 | dataset_index={sample_indices[i]}")
    print(f"Q: {ex['question'][:150]}")
    retrieved = retrieve(ex['question'], ex['documents'], CONFIG)
    print(f"Retrieved {len(retrieved)} chunks | sentences: {[len(c) for c in retrieved]}")
    ans = generate_answer(ex['question'], retrieved)
    print(f"A: {ans[:200]}")
    scores = judge_and_score(ex['question'], ans, retrieved)
    print(f"TRACe → rel={scores['relevance']:.3f} util={scores['utilization']:.3f} "
          f"comp={scores['completeness']:.3f} adh={scores['adherence']:.1f} "
          f"({scores['n_relevant']}rel/{scores['n_utilized']}util/{scores['n_total_sents']}total)")
print("\n✅ Smoke test complete — inspect above before running Cell 12.")


SMOKE 1/3 | dataset_index=12
Q: The date of the contract
Retrieved 5 chunks | sentences: [3, 3, 3, 3, 3]
A: The context does not provide a specific date for the contract, only referring to the "Effective Date" as the starting point of the Initial Term.
TRACe → rel=0.267 util=0.067 comp=0.250 adh=1.0 (4rel/1util/15total)

SMOKE 2/3 | dataset_index=13
Q: What is the renewal term after the initial term expires? This includes automatic extensions and unilateral extensions with prior notice.
Retrieved 5 chunks | sentences: [3, 3, 3, 3, 3]
A: There is no information provided in the context about the renewal term or the initial term. The context only discusses the extension of application, licensing rights, and reversion of license, but doe
TRACe → rel=0.000 util=0.000 comp=0.000 adh=1.0 (0rel/0util/15total)

SMOKE 3/3 | dataset_index=15
Q: Is a party’s liability uncapped upon the breach of its obligation in the contract? This also includes uncap liability for a particular type of breach 
Re

## Cell 12 — Full run (N=25) with resume

In [16]:
results, done = load_progress()

for pos, (idx, ex) in enumerate(zip(sample_indices, sample)):
    if idx in done:
        continue
    print(f"[{pos+1}/{len(sample)}] idx={idx}")
    try:
        retrieved = retrieve(ex['question'], ex['documents'], CONFIG)
        ans = generate_answer(ex['question'], retrieved)
        scores = judge_and_score(ex['question'], ans, retrieved)
    except Exception as e:
        print(f"  ❌ FAILED idx={idx}: {type(e).__name__}: {str(e)[:300]}")
        raise   # full visibility — never swallow
    results.append({
        "experiment_id": CONFIG["experiment_id"],
        "dataset_index": idx,
        "question": ex["question"],
        "answer": ans,
        **{k2: scores[k2] for k2 in ("relevance","utilization","completeness","adherence",
                                       "n_relevant","n_utilized","n_total_sents")},
        "judge_raw": scores["judge_raw"],
    })
    done.add(idx)
    if len(results) % CONFIG["checkpoint_every"] == 0:
        checkpoint(results)

checkpoint(results)
print(f"\n✅ Run complete: {len(results)} examples. Verify aggregates (Cell 13) BEFORE pushing to GitHub.")

[1/25] idx=12
[2/25] idx=13
[3/25] idx=15
[4/25] idx=16
[5/25] idx=44
  ✓ checkpointed 5 results
[6/25] idx=47
[7/25] idx=52
[8/25] idx=57
[9/25] idx=71
[10/25] idx=111
  ✓ checkpointed 10 results
[11/25] idx=114
[12/25] idx=119
[13/25] idx=125
[14/25] idx=140
[15/25] idx=216
  ✓ checkpointed 15 results
[16/25] idx=258
[17/25] idx=279
[18/25] idx=287
[19/25] idx=302
[20/25] idx=308
  ✓ checkpointed 20 results
[21/25] idx=327
[22/25] idx=346
[23/25] idx=377
[24/25] idx=379
[25/25] idx=456
  ✓ checkpointed 25 results
  ✓ checkpointed 25 results

✅ Run complete: 25 examples. Verify aggregates (Cell 13) BEFORE pushing to GitHub.


## Cell 13 — Aggregate TRACe + push (only after verifying)

In [19]:
import statistics as st
print(f"{CONFIG['experiment_id']} | N={len(results)}")
for m in ("relevance", "utilization", "completeness", "adherence"):
    vals = [r[m] for r in results]
    print(f"  {m:13s}: mean={st.mean(vals):.4f}  median={st.median(vals):.4f}")

# After verifying the numbers look sane, push:
checkpoint(results, push_github=True)

LEGAL-EXP-001 | N=25
  relevance    : mean=0.2747  median=0.2667
  utilization  : mean=0.1013  median=0.0667
  completeness : mean=0.3740  median=0.3333
  adherence    : mean=0.8000  median=1.0000
  ✓ checkpointed 25 results + pushed


## Cell 14 — Experiment harness (config-driven, same pattern as CS notebook)
`run(exp_id, **overrides)` — one call per experiment, resumes from checkpoint, aggregates into `ALL_SCORES`.

In [13]:
# ============ COMPONENT REGISTRIES ============
EMBEDDERS = {
    # name: (hf_model, query_prefix, passage_prefix)
    "minilm":    ("sentence-transformers/all-MiniLM-L6-v2", "", ""),
    "e5-large":  ("intfloat/e5-large-v2", "query: ", "passage: "),   # verify prefixes vs locked Biomedical notebook
    "gte-large": ("thenlper/gte-large", "", ""),
    "bge-large": ("BAAI/bge-large-en-v1.5", "Represent this sentence for searching relevant passages: ", ""),
}
_embedder_cache = {}

# model strings verified against CS notebook tracker
GENERATORS = {"70b": "llama-3.3-70b-versatile",
              "8b": "llama-3.1-8b-instant",
              "gpt-oss": "openai/gpt-oss-120b"}
JUDGES     = {"70b": "llama-3.3-70b-versatile",
              "8b": "llama-3.1-8b-instant"}

def get_embedder(name):
    if name not in _embedder_cache:
        model_id, qp, pp = EMBEDDERS[name]
        print(f"  loading embedder: {model_id}")
        m = SentenceTransformer(model_id)
        _embedder_cache[name] = (m, qp, pp, m.get_sentence_embedding_dimension())
    return _embedder_cache[name]

# --- clause chunker (other team's design: one regex, sentence fallback) ---
import re as _re
CLAUSE_SPLIT = _re.compile(
    r"(?=\bARTICLE\s+(?:[IVXLC]+|\d+)\b)"
    r"|(?=\bSection\s+\d+(?:\.\d+)*\b)"
    r"|(?=(?<=\s)\d+(?:\.\d+)+\s+[A-Z])"          # 1.1 Definitions — inline ok
    r"|(?=(?<=\s)\d+\.\s+[A-Z]{2,})",             # 7. CONFIDENTIALITY — inline, needs CAPS word
    _re.IGNORECASE)

def chunk_clause(documents, cfg, max_sents=12):
    """Split on clause boundaries; fall back to sentence chunking if no structure.
    Oversized clauses are sub-split so no chunk exceeds max_sents sentences."""
    chunks = []
    for doc in documents:
        parts = [p.strip() for p in CLAUSE_SPLIT.split(doc) if p and p.strip()]
        if len(parts) < 3:   # no real structure found -> sentence fallback
            chunks.extend(chunk_documents([doc], {**CONFIG, "chunking": "sentence",
                                                   "chunk_size": cfg.get("chunk_size", 3),
                                                   "chunk_overlap": cfg.get("chunk_overlap", 1)}))
            continue
        for part in parts:
            sents = sent_tokenize(part)
            if not sents:
                continue
            if len(sents) <= max_sents:
                chunks.append(sents)
            else:   # giant clause -> sub-split, keep clause header context
                for i in range(0, len(sents), max_sents):
                    chunks.append(sents[i:i + max_sents])
    return chunks

def chunk_dispatch(documents, cfg):
    if cfg["chunking"] == "sentence":
        return chunk_documents(documents, {**CONFIG, "chunking": "sentence",
                                            "chunk_size": cfg.get("chunk_size", 3),
                                            "chunk_overlap": cfg.get("chunk_overlap", 1)})
    elif cfg["chunking"] in ("clause", "sac"):
        return chunk_clause(documents, cfg)
    else:
        raise ValueError(f"Unknown chunking: {cfg['chunking']}")

def retrieve_cfg(question, documents, cfg):
    model, qp, pp, dim = get_embedder(cfg["embedding"])
    chunks = chunk_dispatch(documents, cfg)
    texts = [pp + " ".join(c) for c in chunks]
    emb = model.encode(texts, normalize_embeddings=True, show_progress_bar=False)
    assert emb.shape[1] == dim, f"Dim mismatch! {emb.shape[1]} vs {dim}"
    if cfg["retrieval"] == "dense":
        index = faiss.IndexFlatIP(dim)
        index.add(emb.astype('float32'))
        q = model.encode([qp + question], normalize_embeddings=True).astype('float32')
        if cfg.get("rerank"):
            # wide-retrieve-then-rerank: pull k_wide candidates, rerank, keep k final
            k_wide = min(cfg.get("k_wide", 12), len(chunks))
            _, idxs = index.search(q, k_wide)
            cands = [chunks[i] for i in idxs[0]]
            return rerank_chunks(question, cands, cfg)[: cfg.get("k", 5)]
        k = min(cfg.get("k", 5), len(chunks))
        _, idxs = index.search(q, k)
        return [chunks[i] for i in idxs[0]]
    elif cfg["retrieval"] == "hybrid":
        # Stage 3 — port hybrid RRF verbatim from locked Finance/GK notebook if TRACe points here
        raise NotImplementedError("Hybrid RRF not ported yet — Stage 3")
    else:
        raise ValueError(f"Unknown retrieval: {cfg['retrieval']}")

_reranker = None
def rerank_chunks(question, cands, cfg):
    """Cross-encoder rerank — same model as locked GK notebook (verified)."""
    global _reranker
    if _reranker is None:
        from sentence_transformers import CrossEncoder
        _reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
        print("  loaded reranker: cross-encoder/ms-marco-MiniLM-L-6-v2")
    pairs = [(question, " ".join(c)) for c in cands]
    scores = _reranker.predict(pairs, show_progress_bar=False)
    order = sorted(range(len(cands)), key=lambda i: -scores[i])
    return [cands[i] for i in order]

# ============ HARNESS ============
ALL_SCORES = {}

def run_experiment(exp_id, cfg, smoke=0):
    """Full N run with resume + checkpointing. smoke=3 → first 3 examples only, no save."""
    results_file = f"{exp_id}_results.json"

    def _load():
        p = os.path.join(CONFIG["drive_dir"], results_file)
        if os.path.exists(p):
            with open(p) as f:
                rs = json.load(f)
            rs = [r for r in rs if r.get("experiment_id") == exp_id
                  and r.get("dataset_index") in set(sample_indices)]
            return rs, {r["dataset_index"] for r in rs}
        return [], set()

    def _save(rs, push=False):
        for p in (os.path.join(CONFIG["local_dir"], results_file),
                  os.path.join(CONFIG["drive_dir"], results_file)):
            with open(p, "w") as f:
                json.dump(rs, f, indent=1)
        if push:
            rp = f"/content/repo/{CONFIG['repo_path']}/{results_file}"
            with open(rp, "w") as f:
                json.dump(rs, f, indent=1)
            !cd /content/repo && git add -A && git commit -m "{exp_id}: {len(rs)} examples" -q && git push -q

    results, done = ([], set()) if smoke else _load()
    todo = list(zip(sample_indices, sample))[:smoke] if smoke else list(zip(sample_indices, sample))
    if not smoke and done:
        print(f"  resuming: {len(done)} done")

    for pos, (idx, ex) in enumerate(todo):
        if idx in done:
            continue
        print(f"  [{pos+1}/{len(todo)}] idx={idx}")
        try:
            retrieved = retrieve_cfg(ex['question'], ex['documents'], cfg)
            ans = generate_answer(ex['question'], retrieved, model=GENERATORS[cfg.get('generator', '70b')])
            scores = judge_and_score(ex['question'], ans, retrieved, judge_model=JUDGES[cfg.get('judge', '70b')])
        except Exception as e:
            print(f"  ❌ FAILED idx={idx}: {type(e).__name__}: {str(e)[:300]}")
            raise
        if smoke:
            print(f"    Q: {ex['question'][:100]}")
            print(f"    A: {ans[:150]}")
            print(f"    rel={scores['relevance']:.3f} util={scores['utilization']:.3f} "
                  f"comp={scores['completeness']:.3f} adh={scores['adherence']:.1f}")
        results.append({"experiment_id": exp_id, "dataset_index": idx,
                        "question": ex["question"], "answer": ans,
                        **{k2: scores[k2] for k2 in ("relevance","utilization","completeness",
                                                      "adherence","n_relevant","n_utilized","n_total_sents")},
                        "judge_raw": scores["judge_raw"]})
        done.add(idx)
        if not smoke and len(results) % CONFIG["checkpoint_every"] == 0:
            _save(results)
            print(f"    ✓ checkpointed {len(results)}")
    if not smoke:
        _save(results)

    import statistics as st
    agg = {m: round(st.mean([r[m] for r in results]), 4)
           for m in ("relevance","utilization","completeness","adherence")}
    print(f"  {exp_id}: {agg}")
    return agg

def run(exp_id, smoke=0, **overrides):
    cfg = {**BASE, **overrides}
    print(f"▶ {exp_id} | {cfg}")
    ALL_SCORES[exp_id] = {"cfg": cfg, "scores": run_experiment(exp_id, cfg, smoke=smoke)}

def scoreboard():
    print(f"{'exp':16s} {'rel':>7s} {'util':>7s} {'comp':>7s} {'adh':>7s}")
    for eid, d in ALL_SCORES.items():
        s = d["scores"]
        print(f"{eid:16s} {s['relevance']:7.4f} {s['utilization']:7.4f} "
              f"{s['completeness']:7.4f} {s['adherence']:7.4f}")

print("Harness ready: run(exp_id, **overrides) | run(exp_id, smoke=3, ...) | scoreboard()")


Harness ready: run(exp_id, **overrides) | run(exp_id, smoke=3, ...) | scoreboard()


## Cell 15 — Experiment plan (run stage by stage; update LOCKED after each stage)

In [14]:
# Cell 15 — Legal experiments (run stage by stage; update BASE/LOCKED after each stage)
BASE = {"chunking": "sentence", "chunk_size": 3, "chunk_overlap": 1,
        "embedding": "minilm", "retrieval": "dense", "k": 5,
        "rerank": False, "generator": "70b", "judge": "70b"}

# Stage 0 — baseline (done: 0.2747/0.1013/0.3740/0.8000; resumes from checkpoint, no API calls)
run("LEGAL-EXP-001")

# Stage 1 — embedding  (TRACe points at retrieval; other team converged on gte-large)
#run("LEGAL-EXP-002", smoke=3, embedding="e5-large")   # smoke first, then rerun without smoke
# run("LEGAL-EXP-002", embedding="e5-large")
# run("LEGAL-EXP-003", embedding="gte-large")
# BASE["embedding"] = ?   # LOCK winner

# BASE["embedding"] = "minilm"   # LOCKED — e5-large no gain at N=25, mirrors CS finding

#Stage 2 — k sweep  (other team: 'tunable top_n dominates'; cheap — no re-embed logic changes)
run("LEGAL-EXP-004", k=7)
run("LEGAL-EXP-005", k=12)
# BASE["k"] = 5   # LOCKED — k-sweep monotonically negative (saturates ~6 rel sents, noise dominates)

# Stage 3 — clause chunking  (other team's design; gate PASSED 5/5; sentence fallback built in)
# run("LEGAL-EXP-006", smoke=3, chunking="clause")
run("LEGAL-EXP-006", chunking="clause")
# BASE["chunking"] = "sentence"   # LOCKED — clause chunking flat (n_relevant 4.0 vs 4.1, +30% context padding)

# Stage 4 — wide-retrieve-then-rerank  (retrieve k_wide=12 -> cross-encoder -> keep k=5)
# PORT-IN required first: rerank_chunks() in Cell 14 needs the GK cross-encoder code
# run("LEGAL-EXP-007", smoke=3, rerank=True, k_wide=15)
run("LEGAL-EXP-007", rerank=True, k_wide=15)

# BASE["rerank"] = False   # LOCKED — wide(15)+cross-encoder rerank flat (0.259 vs 0.275); elimination complete

# Stage 4b — combined (ONLY if EXP-006 and EXP-007 both win individually)
# run("LEGAL-EXP-008", chunking="clause", rerank=True, k_wide=12)

# Parked — hybrid RRF (revisit only if TRACe still points at ranking after the above)
# run("LEGAL-EXP-009", retrieval="hybrid")

# Stage 5 — winner confirmation at higher N before locking (R4 false-positive lesson)

# Stage 6 — GENERATION sweep on locked baseline (only untested stage; gpt-oss won CS 0.40->0.48 adh)
# run("LEGAL-EXP-010", smoke=3, generator="gpt-oss")
run("LEGAL-EXP-010", generator="gpt-oss")
# run("LEGAL-EXP-011", generator="8b")   # done: adh 0.68 — cites more, grounds worse
BASE["generator"] = "gpt-oss"   # LOCKED — 0.92/0.53/0.19 sweeps all metrics; decommission-proof; ordering matches CS

# Stage 7 — judge sanity on winner config (NOT comparable to other rows, mirrors CS-EXP-013)
run("LEGAL-EXP-012", judge="8b")

# Stage 8 — transfer validation: rerank under the winning generator (mirrors CS-VAL-001 spirit)
run("LEGAL-EXP-013", generator="gpt-oss", rerank=True, k_wide=15)

# Stage 9 — FINAL N=200 on locked config


scoreboard()

▶ LEGAL-EXP-001 | {'chunking': 'sentence', 'chunk_size': 3, 'chunk_overlap': 1, 'embedding': 'minilm', 'retrieval': 'dense', 'k': 5, 'rerank': False, 'generator': '70b', 'judge': '70b'}
  resuming: 25 done
  LEGAL-EXP-001: {'relevance': 0.2747, 'utilization': 0.1013, 'completeness': 0.374, 'adherence': 0.8}
▶ LEGAL-EXP-004 | {'chunking': 'sentence', 'chunk_size': 3, 'chunk_overlap': 1, 'embedding': 'minilm', 'retrieval': 'dense', 'k': 7, 'rerank': False, 'generator': '70b', 'judge': '70b'}
  resuming: 25 done
  LEGAL-EXP-004: {'relevance': 0.2971, 'utilization': 0.099, 'completeness': 0.3227, 'adherence': 0.72}
▶ LEGAL-EXP-005 | {'chunking': 'sentence', 'chunk_size': 3, 'chunk_overlap': 1, 'embedding': 'minilm', 'retrieval': 'dense', 'k': 12, 'rerank': False, 'generator': '70b', 'judge': '70b'}
  resuming: 25 done
  LEGAL-EXP-005: {'relevance': 0.1768, 'utilization': 0.0845, 'completeness': 0.4177, 'adherence': 0.72}
▶ LEGAL-EXP-006 | {'chunking': 'clause', 'chunk_size': 3, 'chunk_over

In [30]:
ex = sample[1]   # idx=13
chunks = chunk_clause(ex['documents'], {"chunk_size": 3, "chunk_overlap": 1})
print(f"{len(chunks)} chunks | sents per chunk: {[len(c) for c in chunks][:20]}")
hits = [i for i, c in enumerate(chunks) if 'renew' in " ".join(c).lower() or 'initial term' in " ".join(c).lower()]
print(f"chunks mentioning renewal/initial term: {hits}")
for i in hits[:3]:
    print(f"\n--- chunk {i} ({len(chunks[i])} sents) ---\n{' '.join(chunks[i])[:400]}")

36 chunks | sents per chunk: [1, 11, 3, 5, 1, 1, 1, 2, 7, 2, 12, 1, 4, 4, 10, 5, 1, 5, 4, 4]
chunks mentioning renewal/initial term: [29]

--- chunk 29 (12 sents) ---
7. CONFIDENTIALITY. Each Party agrees to hold in confidence confidential information acquired in the course of this relationship with the other Parties and their associates. Each Party agrees to refrain from, either during period of this Agreement or at any other time thereafter, disclosing, using or disseminating such confidential information, for its or another's benefit, in any way acquired in 


## Cell 16 — Results table (team tracker format: scores + RMSE vs RAGBench ground truth + Adh AUCROC)

In [19]:
import pandas as pd
import math

# --- ground-truth lookup from RAGBench annotations (per dataset_index) ---
GT_FIELDS = {"relevance": "relevance_score", "utilization": "utilization_score",
             "completeness": "completeness_score", "adherence": "adherence_score"}

missing = [v for v in GT_FIELDS.values() if v not in sample.column_names]
if missing:
    print(f"⚠️ GT fields not found: {missing}\nAvailable: {sample.column_names}")

GT = {}
for idx, ex in zip(sample_indices, sample):
    GT[idx] = {m: ex.get(f) for m, f in GT_FIELDS.items()}
# include the N=200 final sample if it exists this session
try:
    for idx, ex in zip(final_indices, final_sample):
        GT[idx] = {m: ex.get(f) for m, f in GT_FIELDS.items()}
    print(f"GT covers {len(GT)} indices (25-sample + 200-final)")
except NameError:
    print(f"GT covers {len(GT)} indices (25-sample only — run Cell 17's sampling for the FINAL row)")

def _rmse(pairs):
    pairs = [(p, g) for p, g in pairs if g is not None]
    return math.sqrt(sum((p - g) ** 2 for p, g in pairs) / len(pairs)) if pairs else None

def _aucroc(pairs):
    pairs = [(p, 1 if g else 0) for p, g in pairs if g is not None]
    if not pairs or len({g for _, g in pairs}) < 2:
        return None
    try:
        from sklearn.metrics import roc_auc_score
        return roc_auc_score([g for _, g in pairs], [p for p, _ in pairs])
    except ImportError:
        pos = [p for p, g in pairs if g == 1]; neg = [p for p, g in pairs if g == 0]
        wins = sum((1.0 if a > b else 0.5 if a == b else 0.0) for a in pos for b in neg)
        return wins / (len(pos) * len(neg))

def _load_results(exp_id):
    p = os.path.join(CONFIG["drive_dir"], f"{exp_id}_results.json")
    with open(p) as f:
        return [r for r in json.load(f) if r.get("experiment_id") == exp_id]

EMB_DISPLAY = {"minilm": "all-MiniLM-L6-v2", "e5-large": "e5-large-v2",
               "gte-large": "gte-large", "bge-large": "bge-large-en-v1.5"}
GEN_DISPLAY = {"70b": "llama-3.3-70b", "8b": "llama-3.1-8b", "gpt-oss": "gpt-oss-120b"}

# fallback configs for sessions where ALL_SCORES is empty (matches run history exactly)
_B = {"chunking": "sentence", "chunk_size": 3, "chunk_overlap": 1, "embedding": "minilm",
      "retrieval": "dense", "k": 5, "rerank": False, "generator": "70b", "judge": "70b"}
KNOWN_CFGS = {
    "LEGAL-EXP-001":   dict(_B),
    "LEGAL-EXP-002":   {**_B, "embedding": "e5-large"},
    "LEGAL-EXP-004":   {**_B, "k": 7},
    "LEGAL-EXP-005":   {**_B, "k": 12},
    "LEGAL-EXP-006":   {**_B, "chunking": "clause"},
    "LEGAL-EXP-007":   {**_B, "rerank": True, "k_wide": 15},
    "LEGAL-EXP-010":   {**_B, "generator": "gpt-oss"},
    "LEGAL-EXP-011":   {**_B, "generator": "8b"},
    "LEGAL-EXP-012":   {**_B, "generator": "gpt-oss", "judge": "8b"},
    "LEGAL-EXP-013":   {**_B, "generator": "gpt-oss", "rerank": True, "k_wide": 15},
    "LEGAL-EXP-FINAL": {**_B, "generator": "gpt-oss"},
}

def results_table(exp_ids=None, notes=None):
    exp_ids = exp_ids or list(ALL_SCORES.keys())
    notes = notes or {}
    rows = []
    for eid in exp_ids:
        cfg = ALL_SCORES.get(eid, {}).get("cfg") or KNOWN_CFGS.get(eid, {})
        p = os.path.join(CONFIG["drive_dir"], f"{eid}_results.json")
        if not os.path.exists(p):
            print(f"  (skipping {eid} — no results file yet)")
            continue
        rs = _load_results(eid)
        cells, refs = {}, {}
        for m in ("relevance", "utilization", "completeness", "adherence"):
            pairs = [(r[m], GT.get(r["dataset_index"], {}).get(m)) for r in rs]
            mean_pred = sum(r[m] for r in rs) / len(rs)
            gts = [g for _, g in pairs if g is not None]
            refs[m] = round(sum(1 if g else 0 for g in gts) / len(gts), 4) if (m == "adherence" and gts) \
                      else (round(sum(gts) / len(gts), 4) if gts else None)
            if m == "adherence":
                auc = _aucroc(pairs)
                cells[m] = f"{mean_pred:.4f}\nAUCROC {auc:.4f}" if auc is not None else f"{mean_pred:.4f}\nAUCROC n/a"
            else:
                rm = _rmse(pairs)
                cells[m] = f"{mean_pred:.4f}\nRMSE {rm:.4f}" if rm is not None else f"{mean_pred:.4f}\nRMSE n/a"
        chunk_disp = {"sentence": f"Sentence ({cfg.get('chunk_size', 3)}s/{cfg.get('chunk_overlap', 1)}o)",
                      "clause": "Clause (regex, sentence fallback)"}.get(cfg.get("chunking"), cfg.get("chunking"))
        retr_disp = {"dense": "FAISS (dense)", "hybrid": "Hybrid RRF"}.get(cfg.get("retrieval"), cfg.get("retrieval"))
        if cfg.get("k", 5) != 5:
            retr_disp += f", k={cfg['k']}"
        rr = f"Cross-encoder ms-marco (wide k={cfg.get('k_wide', 12)})" if cfg.get("rerank") else "None"
        gen_disp = GEN_DISPLAY.get(cfg.get("generator"), cfg.get("generator"))
        if cfg.get("judge", "70b") != "70b":
            gen_disp += f" (judge={cfg['judge']})"
        rows.append({
            "Experiment ID": eid, "Domain": "Legal (cuad)",
            "Chunking Strategy": chunk_disp, "Embedding Model": EMB_DISPLAY.get(cfg.get("embedding"), cfg.get("embedding")),
            "Retrieval Technique": retr_disp, "Re-ranking": rr, "Generator LLM Used": gen_disp,
            "Context Relevance (score) / Rel RMSE": cells["relevance"],
            "Context Utilization (score) / Util RMSE": cells["utilization"],
            "Completeness (score) / Comp RMSE": cells["completeness"],
            "Adherence (score) / Adh AUCROC": cells["adherence"],
            "Reference Relevance": refs["relevance"], "Reference Utilization": refs["utilization"],
            "Reference Completeness": refs["completeness"], "Reference Adherence": refs["adherence"],
            "Samples": len(rs), "Notes / Error Analysis": notes.get(eid, ""),
        })
    df = pd.DataFrame(rows)
    out_csv = os.path.join(CONFIG["drive_dir"], "LEGAL_results_table.csv")
    df.to_csv(out_csv, index=False)
    print(f"Saved → {out_csv}")
    return df

results_table(
    exp_ids=["LEGAL-EXP-001","LEGAL-EXP-002","LEGAL-EXP-004","LEGAL-EXP-005","LEGAL-EXP-006",
             "LEGAL-EXP-007","LEGAL-EXP-010","LEGAL-EXP-011","LEGAL-EXP-012","LEGAL-EXP-013","LEGAL-EXP-FINAL"],
    notes={
        "LEGAL-EXP-001": "Baseline. Retrieval-starved: high adh, low rel/util. Adh failures = hedge-elaborations, not confabulation.",
        "LEGAL-EXP-002": "e5-large: no gain (rel 0.27→0.24). Mirrors CS — large embedders don't help.",
        "LEGAL-EXP-004": "k=7: +50% relevant sents retrieved, comp/adh dropped — generator drowns in fragments.",
        "LEGAL-EXP-005": "k=12: retrieval saturates ~6 rel sents. k-sweep monotonically negative → k=5 LOCKED.",
        "LEGAL-EXP-006": "Clause chunking flat (n_rel 4.0 vs 4.1, +30% padding). Chunk shape not the bottleneck.",
        "LEGAL-EXP-007": "Wide(15)+cross-encoder rerank (=GK model): flat. Retrieval stage fully eliminated.",
        "LEGAL-EXP-010": "WINNER + LOCKED. gpt-oss sweeps all metrics (adh 0.92 = GT ref; util 2x; comp 0.53). Generation was the bottleneck. Decommission-proof.",
        "LEGAL-EXP-011": "8b generator: adh 0.68 worst recorded — cites more, grounds worse. Ordering gpt-oss>70b>8b matches CS.",
        "LEGAL-EXP-012": "JUDGE CHECK — NOT comparable. 8b judge scores same pipeline 0.32 vs 0.92 (70b); malformed JSON required salvage parsing. 70b judge requirement confirmed.",
        "LEGAL-EXP-013": "Transfer validation: rerank under winning generator. adh 0.84 vs 0.92 — no interaction; reranker eliminated under both generators.",
        "LEGAL-EXP-FINAL": "AUTHORITATIVE N=200 FINAL on locked config. Adh 0.85 vs GT ref 0.905. AUCROC 0.533 (19 GT-negatives) — near-0.5 structurally expected from label transfer, mirrors GK 0.487. Config selected via controlled N=25 comparisons.",
    })

GT covers 200 indices (25-sample + 200-final)
Saved → /content/drive/MyDrive/RAGBench_Capstone/legal/LEGAL_results_table.csv


,Experiment ID,Domain,Chunking Strategy,Embedding Model,Retrieval Technique,Re-ranking,Generator LLM Used,Context Relevance (score) / Rel RMSE,Context Utilization (score) / Util RMSE,Completeness (score) / Comp RMSE,Adherence (score) / Adh AUCROC,Reference Relevance,Reference Utilization,Reference Completeness,Reference Adherence,Samples,Notes / Error Analysis
0,LEGAL-EXP-001,Legal (cuad),Sentence (3s/1o),all-MiniLM-L6-v2,FAISS (dense),None,llama-3.3-70b,0.2747\nRMSE 0.3760,0.1013\nRMSE 0.1233,0.3740\nRMSE 0.6186,0.8000\nAUCROC 0.9348,0.0756,0.0194,0.7466,0.920,25,"Baseline. Retrieval-starved: high adh, low rel..."
1,LEGAL-EXP-002,Legal (cuad),Sentence (3s/1o),e5-large-v2,FAISS (dense),None,llama-3.3-70b,0.2427\nRMSE 0.3722,0.1280\nRMSE 0.2081,0.4456\nRMSE 0.6120,0.7600\nAUCROC 0.9130,0.0756,0.0194,0.7466,0.920,25,e5-large: no gain (rel 0.27→0.24). Mirrors CS ...
2,LEGAL-EXP-004,Legal (cuad),Sentence (3s/1o),all-MiniLM-L6-v2,"FAISS (dense), k=7",None,llama-3.3-70b,0.2971\nRMSE 0.4124,0.0990\nRMSE 0.1388,0.3227\nRMSE 0.6069,0.7200\nAUCROC 0.8913,0.0756,0.0194,0.7466,0.920,25,"k=7: +50% relevant sents retrieved, comp/adh d..."
3,LEGAL-EXP-005,Legal (cuad),Sentence (3s/1o),all-MiniLM-L6-v2,"FAISS (dense), k=12",None,llama-3.3-70b,0.1768\nRMSE 0.2589,0.0845\nRMSE 0.1266,0.4177\nRMSE 0.6404,0.7200\nAUCROC 0.6196,0.0756,0.0194,0.7466,0.920,25,k=12: retrieval saturates ~6 rel sents. k-swee...
4,LEGAL-EXP-006,Legal (cuad),"Clause (regex, sentence fallback)",all-MiniLM-L6-v2,FAISS (dense),None,llama-3.3-70b,0.2566\nRMSE 0.3487,0.1086\nRMSE 0.1556,0.3637\nRMSE 0.6163,0.7600\nAUCROC 0.3696,0.0756,0.0194,0.7466,0.920,25,"Clause chunking flat (n_rel 4.0 vs 4.1, +30% p..."
5,LEGAL-EXP-007,Legal (cuad),Sentence (3s/1o),all-MiniLM-L6-v2,FAISS (dense),Cross-encoder ms-marco (wide k=15),llama-3.3-70b,0.2587\nRMSE 0.3735,0.1093\nRMSE 0.1519,0.3990\nRMSE 0.5513,0.8000\nAUCROC 0.6630,0.0756,0.0194,0.7466,0.920,25,Wide(15)+cross-encoder rerank (=GK model): fla...
6,LEGAL-EXP-010,Legal (cuad),Sentence (3s/1o),all-MiniLM-L6-v2,FAISS (dense),None,gpt-oss-120b,0.3253\nRMSE 0.4201,0.1947\nRMSE 0.2739,0.5326\nRMSE 0.5096,0.9200\nAUCROC 1.0000,0.0756,0.0194,0.7466,0.920,25,WINNER + LOCKED. gpt-oss sweeps all metrics (a...
7,LEGAL-EXP-011,Legal (cuad),Sentence (3s/1o),all-MiniLM-L6-v2,FAISS (dense),None,llama-3.1-8b,0.3360\nRMSE 0.4482,0.1387\nRMSE 0.1778,0.4026\nRMSE 0.5699,0.6800\nAUCROC 0.8696,0.0756,0.0194,0.7466,0.920,25,8b generator: adh 0.68 worst recorded — cites ...
8,LEGAL-EXP-012,Legal (cuad),Sentence (3s/1o),all-MiniLM-L6-v2,FAISS (dense),None,gpt-oss-120b (judge=8b),0.4613\nRMSE 0.4638,0.2427\nRMSE 0.3127,0.5592\nRMSE 0.5701,0.3200\nAUCROC 0.6739,0.0756,0.0194,0.7466,0.920,25,JUDGE CHECK — NOT comparable. 8b judge scores ...
9,LEGAL-EXP-013,Legal (cuad),Sentence (3s/1o),all-MiniLM-L6-v2,FAISS (dense),Cross-encoder ms-marco (wide k=15),gpt-oss-120b,0.3040\nRMSE 0.3810,0.1733\nRMSE 0.2384,0.5670\nRMSE 0.5219,0.8400\nAUCROC 0.6848,0.0756,0.0194,0.7466,0.920,25,Transfer validation: rerank under winning gene...


In [20]:
import glob, shutil
repo_dir = f"/content/repo/{CONFIG['repo_path']}"
nb_dir = "/content/repo/notebooks"
os.makedirs(nb_dir, exist_ok=True)

# results + tracker
for f in glob.glob(os.path.join(CONFIG["drive_dir"], "LEGAL-EXP-*_results.json")) + \
         glob.glob(os.path.join(CONFIG["drive_dir"], "LEGAL_results_table.csv")):
    shutil.copy(f, repo_dir)
    print("staged:", os.path.basename(f))

# notebook (adjust filename if you named it differently on Drive)
NB = os.path.join(CONFIG["drive_dir"], "RAGBench_Legal_CUAD_Pipeline.ipynb")
if os.path.exists(NB):
    shutil.copy(NB, nb_dir)
    print("staged notebook:", os.path.basename(NB))
else:
    print("⚠️ notebook not found on Drive — upload it there first or push via GitHub web UI")

!cd /content/repo && git config user.name "veenulearns-lab" && git add -A && git commit -m "Legal LOCKED: N=200 FINAL (adh 0.85, AUCROC 0.533) + full 11-exp tracker + notebook. Config: sentence/MiniLM/dense-k5/gpt-oss/70b-judge" -q && git push -q
!cd /content/repo && git log --oneline -2

staged: LEGAL-EXP-002_results.json
staged: LEGAL-EXP-011_results.json
staged: LEGAL-EXP-006_results.json
staged: LEGAL-EXP-010_results.json
staged: LEGAL-EXP-005_results.json
staged: LEGAL-EXP-007_results.json
staged: LEGAL-EXP-001_results.json
staged: LEGAL-EXP-004_results.json
staged: LEGAL-EXP-012_results.json
staged: LEGAL-EXP-013_results.json
staged: LEGAL-EXP-FINAL_results.json
staged: LEGAL_results_table.csv
⚠️ notebook not found on Drive — upload it there first or push via GitHub web UI
dbdc1aa (HEAD -> main, origin/main, origin/HEAD) Legal LOCKED: N=200 FINAL (adh 0.85, AUCROC 0.533) + full 11-exp tracker + notebook. Config: sentence/MiniLM/dense-k5/gpt-oss/70b-judge
04f73fb Legal: EXP-007/010/011/012 + tracker. gpt-oss WINNER (adh 0.92), generator locked; 8b judge check confirms 70b requirement


In [15]:
# ============ Cell 17 — FINAL N=200 on LOCKED config (LEGAL-EXP-FINAL) ============
# Locked: sentence 3/1 + MiniLM + dense k=5 + gpt-oss generator + 70b judge, no rerank
N_FINAL = 200
rng_final = random.Random(CONFIG["SEED"])
final_indices = sorted(rng_final.sample(range(len(ds)), N_FINAL))
final_sample = ds.select(final_indices)
print(f"Final sample: N={N_FINAL} at SEED={CONFIG['SEED']} | first 5: {final_indices[:5]}")

# temporarily point the harness at the 200-sample set
sample_indices_25, sample_25 = sample_indices, sample     # stash iteration set
sample_indices, sample = final_indices, final_sample

run("LEGAL-EXP-FINAL", generator="gpt-oss")

sample_indices, sample = sample_indices_25, sample_25     # restore

Final sample: N=200 at SEED=42 | first 5: [0, 1, 3, 5, 9]
▶ LEGAL-EXP-FINAL | {'chunking': 'sentence', 'chunk_size': 3, 'chunk_overlap': 1, 'embedding': 'minilm', 'retrieval': 'dense', 'k': 5, 'rerank': False, 'generator': 'gpt-oss', 'judge': '70b'}
  [1/200] idx=0
  loading embedder: sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

/tmp/ipykernel_486/4122330264.py:23: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  _embedder_cache[name] = (m, qp, pp, m.get_sentence_embedding_dimension())


  [2/200] idx=1
  [3/200] idx=3
  [4/200] idx=5
  [5/200] idx=9
    ✓ checkpointed 5
  [6/200] idx=12
  [7/200] idx=13
  [8/200] idx=15
  [9/200] idx=16
  [10/200] idx=22
    ✓ checkpointed 10
  [11/200] idx=23
  [12/200] idx=24
  [13/200] idx=28
  [14/200] idx=32
  [15/200] idx=33
    ✓ checkpointed 15
  [16/200] idx=35
  [17/200] idx=36
  [18/200] idx=40
  [19/200] idx=44
  [20/200] idx=46
    ✓ checkpointed 20
  [21/200] idx=47
  [22/200] idx=49
  [23/200] idx=51
  [24/200] idx=52
  [25/200] idx=54
    ✓ checkpointed 25
  [26/200] idx=56
  [27/200] idx=57
  [28/200] idx=58
  [29/200] idx=63
  [30/200] idx=70
    ✓ checkpointed 30
  [31/200] idx=71
  [32/200] idx=73
  [33/200] idx=78
  [34/200] idx=79
  [35/200] idx=80
    ✓ checkpointed 35
  [36/200] idx=81
  [37/200] idx=82
  [38/200] idx=83
  [39/200] idx=87
  [40/200] idx=91
    ✓ checkpointed 40
  [41/200] idx=98
  [429] rotating to key 2, attempt 1
  [42/200] idx=101
  [43/200] idx=107
  [44/200] idx=108
  [45/200] idx=110
    

In [18]:
# ============ FINAL row: GT on the 200 indices + AUCROC ============
GT_FINAL = {}
for idx, ex in zip(final_indices, final_sample):
    GT_FINAL[idx] = {m: ex.get(f) for m, f in GT_FIELDS.items()}

rs = _load_results("LEGAL-EXP-FINAL")
print(f"N = {len(rs)}")
for m in ("relevance", "utilization", "completeness"):
    pairs = [(r[m], GT_FINAL[r["dataset_index"]][m]) for r in rs]
    print(f"{m:13s}: score={sum(p for p,_ in pairs)/len(pairs):.4f}  RMSE={_rmse(pairs):.4f}  ref={sum(g for _,g in pairs if g is not None)/len(pairs):.4f}")
pairs = [(r["adherence"], GT_FINAL[r["dataset_index"]]["adherence"]) for r in rs]
auc = _aucroc(pairs)
gt_pos = sum(1 for _, g in pairs if g)
print(f"adherence    : score={sum(p for p,_ in pairs)/len(pairs):.4f}  AUCROC={auc:.4f}  ref={gt_pos/len(pairs):.4f}  (GT negatives: {len(pairs)-gt_pos})")

N = 200
relevance    : score=0.3595  RMSE=0.4393  ref=0.0986
utilization  : score=0.1886  RMSE=0.2616  ref=0.0460
completeness : score=0.4793  RMSE=0.5779  ref=0.7567
adherence    : score=0.8500  AUCROC=0.5334  ref=0.9050  (GT negatives: 19)
